In [1]:
!wget https://download.nrg.wustl.edu/data/OAS2_RAW_PART1.tar.gz -O oasis_disc.tar.gz

--2025-05-12 15:42:14--  https://download.nrg.wustl.edu/data/OAS2_RAW_PART1.tar.gz
Resolving download.nrg.wustl.edu (download.nrg.wustl.edu)... 128.252.217.126
Connecting to download.nrg.wustl.edu (download.nrg.wustl.edu)|128.252.217.126|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10874461059 (10G) [application/x-gzip]
Saving to: ‘oasis_disc.tar.gz’

oasis_disc.tar.gz   100%[===================>]  10.13G  81.8MB/s    in 3m 22s  

2025-05-12 15:45:36 (51.3 MB/s) - ‘oasis_disc.tar.gz’ saved [10874461059/10874461059]



In [3]:
import os
import nibabel as nib
import numpy as np
import cv2
import zipfile
from tqdm import tqdm
import warnings

# Enhanced Configuration
INPUT_DIR = "/kaggle/working/disc12"
OUTPUT_DIR = "/kaggle/working/oasis_448_pngs" 
ZIP_PATH = "/kaggle/working/oasis12_448.zip"
TARGET_SIZE = (448, 448)  # Keep at 448 for now (can be resized later in dataloader)
PNG_QUALITY = 1           # Near-lossless (0-9)
SAFE_THRESHOLD = 100      # Minimum non-zero pixels
# For this task, we want 1 image per subject (mid-slice only)
SLICES_PER_SUBJECT = 1

def safe_enhance(slice_img):
    """Robust enhancement with validation"""
    # 1. Check for empty slices
    valid_pixels = slice_img[slice_img > np.percentile(slice_img, 5)]
    if len(valid_pixels) < SAFE_THRESHOLD:
        return None
    
    # 2. Dynamic range expansion
    p_low, p_high = np.percentile(valid_pixels, [0.5, 99.5])
    if p_high <= p_low:
        return None
    enhanced = np.clip(slice_img, p_low, p_high)
    enhanced = ((enhanced - p_low) / (p_high - p_low) * 255).astype(np.uint8)
    
    # 3. Edge-preserving enhancement
    blurred = cv2.GaussianBlur(enhanced, (0,0), 2)
    sharpened = cv2.addWeighted(enhanced, 1.7, blurred, -0.7, 0)
    
    # 4. CLAHE with optimized params
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(16,16))
    return clahe.apply(sharpened)

def locate_scan(subject_path):
    """
    Locate the scan file by giving priority to the T88_111 folder if it exists.
    If not found, fall back to the previous search method.
    """
    # First try T88_111 folder
    t88_folder = os.path.join(subject_path, "T88_111")
    if os.path.exists(t88_folder):
        for f in os.listdir(t88_folder):
            if f.endswith(".nii") or f.endswith(".nii.gz"):
                return os.path.join(t88_folder, f)
    
    # Fallback: use the previous search paths (e.g., SUBJ_111)
    search_paths = [
        os.path.join(subject_path, "PROCESSED", "MPRAGE", "SUBJ_111"),
        os.path.join(subject_path, "PROCESSED", "SUBJ_111"),
        os.path.join(subject_path, "ANALYZE"),
        subject_path  # Fallback to root if nothing else found
    ]
    
    for path in search_paths:
        if os.path.exists(path):
            for f in os.listdir(path):
                if (f.endswith(".nii") or f.endswith(".nii.gz") or f.endswith(".img")) and ("mpr" in f.lower()):
                    return os.path.join(path, f)
    return None

def process_subject(subject_path):
    try:
        scan_path = locate_scan(subject_path)
        if not scan_path:
            return []
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            data = nib.load(scan_path).get_fdata()
        
        # For classification, we choose the mid slice only.
        mid = data.shape[2] // 2
        # Use only the mid slice for simplicity in classification
        slice_indices = [mid] if SLICES_PER_SUBJECT == 1 else [mid-1, mid, mid+1][:SLICES_PER_SUBJECT]
        
        processed = []
        for i in slice_indices:
            if 0 <= i < data.shape[2]:
                slice_img = data[:, :, i]
                enhanced = safe_enhance(slice_img)
                if enhanced is not None:
                    # Resize using dual-phase resizing based on dimensions
                    if enhanced.shape[0] > TARGET_SIZE[0]:
                        resized = cv2.resize(enhanced, TARGET_SIZE, interpolation=cv2.INTER_AREA)
                    else:
                        resized = cv2.resize(enhanced, TARGET_SIZE, interpolation=cv2.INTER_LANCZOS4)
                    # Create an RGB image by merging three channels
                    processed.append(cv2.merge([resized]*3))
        return processed
    
    except Exception as e:
        print(f"Skipped {os.path.basename(subject_path)}: {str(e)}")
        return []

# Main Processing
os.makedirs(OUTPUT_DIR, exist_ok=True)
subjects = [d for d in os.listdir(INPUT_DIR) if d.startswith("OAS1") and os.path.isdir(os.path.join(INPUT_DIR, d))]

print(f"Processing {len(subjects)} subjects at {TARGET_SIZE} resolution...")
success_count = 0
all_png_files = []
for subject in tqdm(subjects):
    slices = process_subject(os.path.join(INPUT_DIR, subject))
    if slices:
        # For classification, we store one representative image per subject.
        # We'll use the first (or only) slice.
        output_path = os.path.join(OUTPUT_DIR, f"{subject}_slice0.png")
        cv2.imwrite(output_path, slices[0], [
            cv2.IMWRITE_PNG_COMPRESSION, PNG_QUALITY,
            cv2.IMWRITE_PNG_STRATEGY, 0
        ])
        all_png_files.append(output_path)
        success_count += 1

# ZIP Creation with Progress
print("\nCreating ZIP archive...")
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zipf:
    with tqdm(total=len(all_png_files), unit='files') as pbar:
        for file in all_png_files:
            arcname = os.path.basename(file)
            zipf.write(file, arcname)
            pbar.update(1)

# Verification
print("\n=== Processing Report ===")
print(f"Successfully processed: {success_count}/{len(subjects)} subjects")
print(f"Total images generated: {len(all_png_files)}")
print(f"ZIP size: {os.path.getsize(ZIP_PATH)/1024/1024:.2f} MB")


Processing 36 subjects at (448, 448) resolution...


  0%|          | 0/36 [00:00<?, ?it/s]very large origin values relative to dims; leaving as set, ignoring for affine
very large origin values relative to dims; leaving as set, ignoring for affine
  3%|▎         | 1/36 [00:00<00:05,  5.89it/s]very large origin values relative to dims; leaving as set, ignoring for affine
very large origin values relative to dims; leaving as set, ignoring for affine
very large origin values relative to dims; leaving as set, ignoring for affine
very large origin values relative to dims; leaving as set, ignoring for affine
  8%|▊         | 3/36 [00:00<00:02, 12.26it/s]very large origin values relative to dims; leaving as set, ignoring for affine
very large origin values relative to dims; leaving as set, ignoring for affine
very large origin values relative to dims; leaving as set, ignoring for affine
very large origin values relative to dims; leaving as set, ignoring for affine
very large origin values relative to dims; leaving as set, ignoring for affine
v


Creating ZIP archive...


100%|██████████| 36/36 [00:00<00:00, 168.43files/s]


=== Processing Report ===
Successfully processed: 36/36 subjects
Total images generated: 36
ZIP size: 5.16 MB
